In [7]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
from scipy.signal import find_peaks

In [9]:
# ---------------------------------------------------
# Load BC-UniGe data and compute peak values
# ---------------------------------------------------
dyn_bc = np.loadtxt("C:/Users/Rodrigo/Desktop/Rodrigo/02 - Research fellow/01 - Storm surge ML downscaling/Predictand/meteoDTNLP13h_bcunige_Trieste_1987-2020.txt")
tg = np.loadtxt("C:/Users/Rodrigo/Desktop/Rodrigo/02 - Research fellow/01 - Storm surge ML downscaling/Predictand/meteoDTNLP13h_TG_Trieste_1987-2020_UTC0.txt")

chunksTest = [1995, 1996, 2020]   # test years
# Find indices of rows where the first column (year) matches chunksTest
id_year_ini_test = np.where(np.isin(tg[:, 0], chunksTest))[0]
# Select test data based on the found indices
tgTest = tg[id_year_ini_test][:-1]  # Remove last row for alignment
# Select corresponding BC-UniGe test data and remove the first row for each variable
dynTest = dyn_bc[id_year_ini_test][1:]  # Shift selection by one row

# Remove the mean
xx = tgTest[:, 7] - np.nanmean(tgTest[:, 7])
yy = dynTest[:, 7] - np.nanmean(dynTest[:, 7])

storm_surge_threshold = np.percentile(xx, 99)
peaks, _ = find_peaks(xx, height=storm_surge_threshold)

x_peaks = xx[peaks]
y_bc_peaks = yy[peaks]

# Percentiles for BC
sta_prc = np.percentile(x_peaks, np.arange(0, 101, 1))
bc_prc = np.percentile(y_bc_peaks, np.arange(0, 101, 1))

# BC metrics
mad_bc = np.nanmean(np.abs(x_peaks - y_bc_peaks))
bias_bc = np.nanmean(y_bc_peaks - x_peaks)
madp_bc = np.nanmean(np.abs(bc_prc - sta_prc))
madc_bc = mad_bc + madp_bc

print(mad_bc)
print(bias_bc)
print(madp_bc)
print(madc_bc)

0.09571888537590748
-0.07291474873052788
0.07502969240561203
0.1707485777815195


In [11]:
# Bootstrap

# ---------------------------
# Parameters
# ---------------------------
n_bootstrap = 10000               # bootstrap iterations
alpha = 0.05                      # significance level

# ---------------------------
# Helper: paired bootstrap for unsigned metrics
# ---------------------------
def paired_bootstrap(errors_bc, errors_em, n_bootstrap, alpha):
    """
    Paired bootstrap test for unsigned metrics
    (MAD, MADp, MADc).

    Returns:
        diff = mean(em) - mean(bc)
        confidence interval
        probability emulator is better
    """

    n = len(errors_bc)
    diffs = []
    rng = np.random.default_rng()

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        sample_diff = np.mean(errors_em[idx]) - np.mean(errors_bc[idx])
        diffs.append(sample_diff)

    diffs = np.array(diffs)

    ci_lower = np.percentile(diffs, 100 * alpha / 2)
    ci_upper = np.percentile(diffs, 100 * (1 - alpha / 2))

    p_em_better = np.mean(diffs < 0)

    return (
        np.mean(errors_em) - np.mean(errors_bc),
        (ci_lower, ci_upper),
        p_em_better
    )


# ---------------------------
# Helper: paired bootstrap for signed bias
# ---------------------------
def paired_bootstrap_bias(res_bc, res_em, n_bootstrap, alpha):
    """
    Bootstrap comparison of BIAS MAGNITUDE.

    Uses:
        |mean(residual_em)| - |mean(residual_bc)|

    Negative values = emulator better.
    """

    n = len(res_bc)
    diffs = []
    rng = np.random.default_rng()

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        bias_bc_sample = np.mean(res_bc[idx])
        bias_em_sample = np.mean(res_em[idx])

        sample_diff = np.abs(bias_em_sample) - np.abs(bias_bc_sample)

        diffs.append(sample_diff)

    diffs = np.array(diffs)

    ci_lower = np.percentile(diffs, 100 * alpha / 2)
    ci_upper = np.percentile(diffs, 100 * (1 - alpha / 2))

    diff = np.abs(np.mean(res_em)) - np.abs(np.mean(res_bc))

    p_em_better = np.mean(diffs < 0)

    return diff, (ci_lower, ci_upper), p_em_better

# ---------------------------
# Error metrics storage
# ---------------------------
results = []

for i in range(1, 41):  # Loop through emulator runs
    dyn = np.loadtxt(f'lstmHybridModel_Run{i}_testresults.txt')

    # Observations and emulator
    x = dyn[:, 8] - np.nanmean(dyn[:, 8])  # Obs
    y_em = dyn[:, 7] - np.nanmean(dyn[:, 7])  # Emulator

    # Identify observed peaks
    storm_surge_threshold = np.percentile(x, 99)
    peaks, _ = find_peaks(x, height=storm_surge_threshold)

    if len(peaks) > 1:
        x_peaks = x[peaks]
        y_em_peaks = y_em[peaks]

        # Raw metrics (direct comparison with obs)
        mad_em = np.nanmean(np.abs(x_peaks - y_em_peaks))
        bias_em = np.nanmean(y_em_peaks - x_peaks)       # signed
        abs_bias_em = np.abs(bias_em)                    # absolute bias

        sta_prc = np.percentile(x_peaks, np.arange(0, 101, 1))
        em_prc  = np.percentile(y_em_peaks, np.arange(0, 101, 1))
        madp_em = np.nanmean(np.abs(em_prc - sta_prc))

        madc_em = mad_em + madp_em

        # ----------------------------------------
        # Bootstrap tests (emulator vs BC-UniGe)
        # ----------------------------------------
        # MAD: per-peak absolute errors
        diff_mad, ci_mad, p_mad = paired_bootstrap(
            np.abs(x_peaks - y_bc_peaks),
            np.abs(x_peaks - y_em_peaks),
            n_bootstrap, alpha
        )

        # Bias: compare absolute bias HERE IS THE MAIN CHANGE COMPARED TO THE 2025 VERSION --> np.abs was removed and new helper added
        diff_bias, ci_bias, p_bias = paired_bootstrap_bias(
            (y_bc_peaks - x_peaks),
            (y_em_peaks - x_peaks),
            n_bootstrap, alpha
        )

        # MADp: per-percentile errors
        diff_madp, ci_madp, p_madp = paired_bootstrap(
            np.abs(bc_prc - sta_prc),
            np.abs(em_prc - sta_prc),
            n_bootstrap, alpha
        )

        # MADc: combine per-peak + per-percentile errors
        bc_errors_madc = np.concatenate([
            np.abs(x_peaks - y_bc_peaks),
            np.abs(bc_prc - sta_prc)
        ])
        em_errors_madc = np.concatenate([
            np.abs(x_peaks - y_em_peaks),
            np.abs(em_prc - sta_prc)
        ])
        diff_madc, ci_madc, p_madc = paired_bootstrap(
            bc_errors_madc, em_errors_madc,
            n_bootstrap, alpha
        )

        # Save results
        results.append([
            i,
            mad_em, mad_bc, diff_mad, ci_mad[0], ci_mad[1], p_mad,
            bias_em, bias_bc, diff_bias, ci_bias[0], ci_bias[1], p_bias,
            madp_em, madp_bc, diff_madp, ci_madp[0], ci_madp[1], p_madp,
            madc_em, madc_bc, diff_madc, ci_madc[0], ci_madc[1], p_madc
        ])
    else:
        print(f"Skipping Run {i}: Not enough peaks.")

# Save to DataFrame
columns = [
    "Run",
    "MAD_em", "MAD_bc", "Diff_MAD", "CI_MAD_low", "CI_MAD_high", "p_MAD",
    "Bias_em", "Bias_bc", "Diff_Bias", "CI_Bias_low", "CI_Bias_high", "p_Bias",
    "MADp_em", "MADp_bc", "Diff_MADp", "CI_MADp_low", "CI_MADp_high", "p_MADp",
    "MADc_em", "MADc_bc", "Diff_MADc", "CI_MADc_low", "CI_MADc_high", "p_MADc"
]
df_results = pd.DataFrame(results, columns=columns)
df_results.to_csv("Bootstrap_with_RawMetrics_2026.txt", sep="\t", float_format="%.3f", index=False)

print("✅ Results saved with corrected bias and MADc bootstrap!")
print(df_results.head())


✅ Results saved with corrected bias and MADc bootstrap!
   Run    MAD_em    MAD_bc  Diff_MAD  CI_MAD_low  CI_MAD_high   p_MAD  \
0    1  0.100526  0.095719  0.004807   -0.029160     0.041775  0.3924   
1    2  0.099181  0.095719  0.003462   -0.026856     0.034640  0.4117   
2    3  0.087117  0.095719 -0.008602   -0.036780     0.022145  0.7235   
3    4  0.079724  0.095719 -0.015995   -0.044287     0.013000  0.8574   
4    5  0.081370  0.095719 -0.014349   -0.041614     0.015220  0.8395   

    Bias_em   Bias_bc  Diff_Bias  ...  Diff_MADp  CI_MADp_low  CI_MADp_high  \
0 -0.048832 -0.072915  -0.024083  ...  -0.022968    -0.030277     -0.014888   
1 -0.040342 -0.072915  -0.032573  ...  -0.018473    -0.024596     -0.012010   
2 -0.072273 -0.072915  -0.000642  ...  -0.005396    -0.014393      0.004359   
3 -0.058506 -0.072915  -0.014408  ...  -0.016773    -0.026105     -0.006391   
4 -0.052786 -0.072915  -0.020129  ...  -0.024973    -0.033084     -0.016130   

   p_MADp   MADc_em   MADc_bc 